# Wildfire Survival Prediction - Data Cleaning & Image Acquisition

This notebook performs the following steps:
1.  **Loads** the raw DINS dataset.
2.  **Cleans** the data by selecting relevant columns and filtering for residential structures.
3.  **Labels** the data into 'Burned' (1) and 'Survived' (0) categories.
4.  **Downloads** satellite imagery for each location using a robust tile downloader.

In [3]:
# Install directly in the notebook kernel
%pip install pandas numpy requests mercantile pillow matplotlib

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 2.2 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 71.7 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 19.1 MB/s eta 0:00:01
     |████████████████████████████████| 7.8 MB 17.2 MB/s eta 0:00:01
     |████████████████████████████████| 347 kB 40.8 MB/s eta 0:00:01
     |████████████████████████████████| 509 kB 22.4 MB/s eta 0:00:01
     |████████████████████████████████| 98 kB 15.2 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 25.3 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 10.6 MB/s eta 0:00:01
     |████████████████████████████████| 2.8 MB 32.9 MB/s eta 0:00:01
     |████████████████████████████████| 113 kB 52.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the k

In [10]:
import pandas as pd
import numpy as np
import requests
import mercantile
import os
import time
from PIL import Image
from io import BytesIO

# Configuration
RAW_DATA_PATH = "../data/raw/dins_raw.csv"
PROCESSED_DATA_DIR = "../data/processed"
IMAGE_DIR = "../data/images"
ZOOM_LEVEL = 19  # High resolution for individual properties

# Ensure directories exist
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

## 1. Data Loading & Initial Cleaning

In [12]:
try:
    df_raw = pd.read_csv(RAW_DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    print("UTF-8 failed, trying ISO-8859-1...")
    df_raw = pd.read_csv(RAW_DATA_PATH, encoding='ISO-8859-1')

print(f"Original Rows: {len(df_raw)}")

# Define essential columns
keep_cols = [
    'AIN',               # Unique ID
    'SitusFullAddress',  # Address (for reference)
    'CENTER_LAT',        # Latitude
    'CENTER_LON',        # Longitude
    '* Damage',          # Target Label
    'Structure Category' # Structure Type
]

# Create clean dataframe
df_clean = df_raw[keep_cols].copy()
df_clean.columns = ['id', 'address', 'lat', 'lon', 'damage_str', 'structure_type']

# Drop rows with missing essential coordinates or labels
df_clean = df_clean.dropna(subset=['lat', 'lon', 'damage_str'])
print(f"Rows after dropping missing coordinates/labels: {len(df_clean)}")

Original Rows: 24461
Rows after dropping missing coordinates/labels: 24452


/var/folders/10/msnyrqnd2n91mr9wdrp8tx2h0000gn/T/ipykernel_24454/313070914.py:2: DtypeWarning: Columns (18,28,35,42,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(RAW_DATA_PATH, encoding='utf-8')


## 2. Filtering Structure Types

In [13]:
# Define valid residential types
valid_types = [
    'Single Residence', 
    'Multiple Residence', 
    'Single Family Residence', 
    'Residential'
]

# Filter
df_clean = df_clean[df_clean['structure_type'].isin(valid_types)]
print(f"Rows after filtering structure types: {len(df_clean)}")

Rows after filtering structure types: 21361


## 3. Label Encoding (Burned vs Survived)
*   **1 (Burned):** Destroyed or Major damage.
*   **0 (Survived):** Minor, Affected, or No Damage.

In [14]:
def map_damage(val):
    if pd.isna(val):
        return np.nan
    
    val = str(val).lower().strip()
    
    # BURNED (1)
    if 'destroyed' in val or 'major' in val:
        return 1
    # SURVIVED (0)
    elif 'minor' in val or 'affected' in val or 'no damage' in val:
        return 0
    # AMBIGUOUS / OTHER
    else:
        return np.nan

df_clean['target'] = df_clean['damage_str'].apply(map_damage)
df_clean = df_clean.dropna(subset=['target'])

print("Target Distribution:")
print(df_clean['target'].value_counts())

Target Distribution:
target
1    11771
0     9590
Name: count, dtype: int64


In [15]:
# Updated Chunk 5: Balance & Save with Safe Filenames
import re

# Separate Burned vs Survived
burned = df_clean[df_clean['target'] == 1]
survived = df_clean[df_clean['target'] == 0]

# Balance (Optional: Uncomment to limit size)
# if len(survived) > len(burned):
#     survived = survived.sample(n=len(burned), random_state=42)

final_df = pd.concat([burned, survived]).sample(frac=1).reset_index(drop=True)

# --- NEW: Create a "safe" filename from the address ---
def clean_filename(addr):
    # invalid chars for filenames: / \ : * ? " < > |
    # We replace spaces with underscores and remove weird chars
    clean = str(addr).upper().strip()
    clean = re.sub(r'[^\w\s-]', '', clean) # Remove punctuation
    clean = re.sub(r'[-\s]+', '_', clean)  # Replace spaces/hyphens with underscore
    return clean

final_df['filename'] = final_df['address'].apply(clean_filename)

# Save
final_df.to_csv("../data/processed/clean_homes.csv", index=False)
print(f"Saved {len(final_df)} homes with filenames like: {final_df['filename'].iloc[0]}.jpg")

Saved 21361 homes with filenames like: 16755_MARQUEZ_TER_LOS_ANGELES_CA_90272.jpg
